In [3]:
!pip3 install requests python-dotenv

In [1]:
import os
import requests
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth
load_dotenv()

True

In [2]:
KROGER_CLIENT_ID = os.getenv("KROGER_CLIENT_ID")
KROGER_CLIENT_SECRET = os.getenv("KROGER_CLIENT_SECRET")
BASE_URL = 'https://api.kroger.com'

In [3]:
def get_access_token():
    url = "https://api.kroger.com/v1/connect/oauth2/token"
    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }
    data = {
        "grant_type": "client_credentials",
        "scope": "product.compact"
    }

    response = requests.post(url, headers=headers, data=data,
                             auth=HTTPBasicAuth(KROGER_CLIENT_ID, KROGER_CLIENT_SECRET))

    print(response.status_code, response.text)  # for debugging
    response.raise_for_status()

    return response.json()["access_token"]


In [4]:
def list_locations(token, zip_code="46207", radius_miles=10):
    url = f"{BASE_URL}/v1/locations"
    headers = {"Authorization": f"Bearer {token}"}
    params = {"filter.zipCode.near": zip_code, "filter.radiusInMiles": radius_miles}
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    locations = response.json()["data"]
    return [(loc["locationId"], loc["chain"], loc["address"]["addressLine1"]) for loc in locations]


In [5]:
def select_location(locations, index=0):
    return locations[index][0]

In [15]:
def search_products(token, store_id, term):
    url = f"{BASE_URL}/v1/products"
    headers = {"Authorization": f"Bearer {token}"}
    params = {
        "filter.term": term,
        "filter.locationId": store_id
        #"filter.limit": 5
    }
    print(url)
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    products = response.json()["data"]
    print(products[0])
    return [{
        "productId": p["productId"],
        "description": p["description"],
        "price": p.get("items", [{}])[0].get("price", {}).get("regular", "N/A")
    } for p in products]

In [7]:
def select_product(products, index=0):
    return products[index]["productId"]

In [8]:
def add_to_cart_placeholder():
    print("🛒 Note: To add items to a real cart, you'll need full user authorization (OAuth2 Authorization Code Flow).")


In [9]:
token = get_access_token()

locations = list_locations(token, zip_code="30301")  # Atlanta ZIP
store_id = select_location(locations)

products = search_products(token, store_id, "eggs")
for i, p in enumerate(products):
    print(f"{i+1}. {p['description']} - ${p['price']}")

product_id = select_product(products)
add_to_cart_placeholder()


200 {"expires_in":1800,"access_token":"eyJhbGciOiJSUzI1NiIsImprdSI6Imh0dHBzOi8vYXBpLmtyb2dlci5jb20vdjEvLndlbGwta25vd24vandrcy5qc29uIiwia2lkIjoiWjRGZDNtc2tJSDg4aXJ0N0xCNWM2Zz09IiwidHlwIjoiSldUIn0.eyJhdWQiOiJ5b3VjYXJ0LTI0MzI2MTI0MzAzNDI0NDU0ODVhNWE0NzcyNzM3MDRhNjI3NzMzNzM2MjUwNDc3YTQ2MzI3MTZiNzU1MDY1MzE1NjM3NzY3YTY5NGI2NzY2NzI2NjYyNDM2NjQyNTE0YzZiNDY2ZTcxNmUzNjZiNjEzNzZjMzQ0MzU0NzcxMDY2NjkzNDE0NzU2NDkiLCJleHAiOjE3NDQ1MDc3MzksImlhdCI6MTc0NDUwNTkzNCwiaXNzIjoiYXBpLmtyb2dlci5jb20iLCJzdWIiOiJjZjA0ZWNiNS1mOGQxLTVlMDMtYTA3ZC02YzYwYTdkNWNkODgiLCJzY29wZSI6InByb2R1Y3QuY29tcGFjdCIsImF1dGhBdCI6MTc0NDUwNTkzOTQxNzE3MzEyNSwiYXpwIjoieW91Y2FydC0yNDMyNjEyNDMwMzQyNDQ1NDg1YTVhNDc3MjczNzA0YTYyNzczMzczNjI1MDQ3N2E0NjMyNzE2Yjc1NTA2NTMxNTYzNzc2N2E2OTRiNjc2NjcyNjY2MjQzNjY0MjUxNGM2YjQ2NmU3MTZlMzY2YjYxMzc2YzM0NDM1NDc3MTA2NjY5MzQxNDc1NjQ5In0.ZbvY1TVM9vs7bKSiusAqplaR3HfRl7u4NMpoZvzcSzWeM6B5QIaAoKzqXAwjOT3_ekMHF1JEFjFMXJiz9oxqYLwgn4PKWyR0ViRPQuWqPNTN9iH81c-YO6AifIfSJuOhrTsqgnEiMzgjSEeZrXTfF9NFrheMMTa0NZNwPgD1jHCRlq70u

In [10]:
token = get_access_token()

200 {"expires_in":1800,"access_token":"eyJhbGciOiJSUzI1NiIsImprdSI6Imh0dHBzOi8vYXBpLmtyb2dlci5jb20vdjEvLndlbGwta25vd24vandrcy5qc29uIiwia2lkIjoiWjRGZDNtc2tJSDg4aXJ0N0xCNWM2Zz09IiwidHlwIjoiSldUIn0.eyJhdWQiOiJ5b3VjYXJ0LTI0MzI2MTI0MzAzNDI0NDU0ODVhNWE0NzcyNzM3MDRhNjI3NzMzNzM2MjUwNDc3YTQ2MzI3MTZiNzU1MDY1MzE1NjM3NzY3YTY5NGI2NzY2NzI2NjYyNDM2NjQyNTE0YzZiNDY2ZTcxNmUzNjZiNjEzNzZjMzQ0MzU0NzcxMDY2NjkzNDE0NzU2NDkiLCJleHAiOjE3NDQ1MDc3NDEsImlhdCI6MTc0NDUwNTkzNiwiaXNzIjoiYXBpLmtyb2dlci5jb20iLCJzdWIiOiJjZjA0ZWNiNS1mOGQxLTVlMDMtYTA3ZC02YzYwYTdkNWNkODgiLCJzY29wZSI6InByb2R1Y3QuY29tcGFjdCIsImF1dGhBdCI6MTc0NDUwNTk0MTA1MDMwNzAzNywiYXpwIjoieW91Y2FydC0yNDMyNjEyNDMwMzQyNDQ1NDg1YTVhNDc3MjczNzA0YTYyNzczMzczNjI1MDQ3N2E0NjMyNzE2Yjc1NTA2NTMxNTYzNzc2N2E2OTRiNjc2NjcyNjY2MjQzNjY0MjUxNGM2YjQ2NmU3MTZlMzY2YjYxMzc2YzM0NDM1NDc3MTA2NjY5MzQxNDc1NjQ5In0.ByXuz2nanRg6voiuoyY8424g1uoMPt9qTSJpm4pyvV8e-4iCTCDIPXFQTEwy6JQgs0oEDYIBZiNQe61eqjrBzKwLAxZYp5xtUJbhPTzUXU7NiACA6vQscjAdgRprfNJLzZ6IRvUOFH7BZJ7yjHCbzc_QQBU2aZ1hcY2opclWY_4KPflTK

In [11]:
locations = list_locations(token, zip_code="76207")

In [12]:
locations

[('03500493', 'KROGER', '500 W University Dr'),
 ('03500586', 'KROGER', '1592 S Loop 288'),
 ('03500570', 'KROGER', '5021 Hickory Creek Rd')]

In [13]:
store_id = select_location(locations)

In [16]:
products = search_products(token, store_id, "eggs")

https://api.kroger.com/v1/products
{'productId': '0001111079770', 'upc': '0001111079770', 'productPageURI': '/p/simple-truth-natural-cage-free-large-brown-eggs/0001111079770?cid=dis.api.tpi_products-api_20240521_b:all_c:p_t:youcart-243261243034', 'aisleLocations': [{'bayNumber': '21', 'description': 'DAIRY', 'number': '100', 'numberOfFacings': '4', 'side': 'L', 'shelfNumber': '2', 'shelfPositionInBay': '1'}], 'brand': 'Simple Truth', 'categories': ['Breakfast', 'Dairy', 'Natural & Organic'], 'countryOrigin': 'UNITED STATES', 'description': 'Simple Truth™ Natural Cage Free Large Brown Eggs', 'images': [{'perspective': 'back', 'sizes': [{'size': 'xlarge', 'url': 'https://www.kroger.com/product/images/xlarge/back/0001111079770'}, {'size': 'large', 'url': 'https://www.kroger.com/product/images/large/back/0001111079770'}, {'size': 'medium', 'url': 'https://www.kroger.com/product/images/medium/back/0001111079770'}, {'size': 'small', 'url': 'https://www.kroger.com/product/images/small/back/00

In [25]:
for i, p in enumerate(products):
    print(f"{i+1}. {p['description']} - ${p['price']}")

1. Simple Truth™ Natural Cage Free Large Brown Eggs - $5.29
2. Eggland's Best Classic Large White Eggs, 18 count - $7.29
3. Kroger® Grade A Large White Eggs - $5.89
4. Happy Egg Co.® Free Range Large Brown Eggs - $9.99
5. Happy Egg Co.® Free Range Large Brown Organic Eggs - $8.29
6. Kroger® Medium Grade A White Eggs - $9.69
7. Simple Truth™ Natural Cage Free Large Brown Eggs - $7.79
8. Kroger® Large White Eggs - $19.49
9. Kroger® Grade A Large Eggs - $3.99
10. Kroger® Extra Large White Eggs - $4.19


In [26]:
{'productId': '0001111079770', 'upc': '0001111079770', 'productPageURI': '/p/simple-truth-natural-cage-free-large-brown-eggs/0001111079770?cid=dis.api.tpi_products-api_20240521_b:all_c:p_t:youcart-243261243034', 'aisleLocations': [{'bayNumber': '21', 'description': 'DAIRY', 'number': '100', 'numberOfFacings': '4', 'side': 'L', 'shelfNumber': '2', 'shelfPositionInBay': '1'}], 'brand': 'Simple Truth', 'categories': ['Breakfast', 'Dairy', 'Natural & Organic'], 'countryOrigin': 'UNITED STATES', 'description': 'Simple Truth™ Natural Cage Free Large Brown Eggs', 'images': [{'perspective': 'back', 'sizes': [{'size': 'xlarge', 'url': 'https://www.kroger.com/product/images/xlarge/back/0001111079770'}, {'size': 'large', 'url': 'https://www.kroger.com/product/images/large/back/0001111079770'}, {'size': 'medium', 'url': 'https://www.kroger.com/product/images/medium/back/0001111079770'}, {'size': 'small', 'url': 'https://www.kroger.com/product/images/small/back/0001111079770'}, {'size': 'thumbnail', 'url': 'https://www.kroger.com/product/images/thumbnail/back/0001111079770'}]}, {'perspective': 'front', 'featured': True, 'sizes': [{'size': 'xlarge', 'url': 'https://www.kroger.com/product/images/xlarge/front/0001111079770'}, {'size': 'large', 'url': 'https://www.kroger.com/product/images/large/front/0001111079770'}, {'size': 'medium', 'url': 'https://www.kroger.com/product/images/medium/front/0001111079770'}, {'size': 'small', 'url': 'https://www.kroger.com/product/images/small/front/0001111079770'}, {'size': 'thumbnail', 'url': 'https://www.kroger.com/product/images/thumbnail/front/0001111079770'}]}], 'items': [{'itemId': '0001111079770', 'inventory': {'stockLevel': 'HIGH'}, 'favorite': False, 'fulfillment': {'curbside': True, 'delivery': True, 'inStore': True, 'shipToHome': False}, 'price': {'regular': 5.49, 'promo': 0}, 'size': '12 ct', 'soldBy': 'UNIT'}], 'itemInformation': {'depth': '4.0', 'height': '2.75', 'width': '11.5'}, 'temperature': {'indicator': 'Refrigerated', 'heatSensitive': False}}

{'productId': '0001111079770',
 'upc': '0001111079770',
 'productPageURI': '/p/simple-truth-natural-cage-free-large-brown-eggs/0001111079770?cid=dis.api.tpi_products-api_20240521_b:all_c:p_t:youcart-243261243034',
 'aisleLocations': [{'bayNumber': '21',
   'description': 'DAIRY',
   'number': '100',
   'numberOfFacings': '4',
   'side': 'L',
   'shelfNumber': '2',
   'shelfPositionInBay': '1'}],
 'brand': 'Simple Truth',
 'categories': ['Breakfast', 'Dairy', 'Natural & Organic'],
 'countryOrigin': 'UNITED STATES',
 'description': 'Simple Truth™ Natural Cage Free Large Brown Eggs',
 'images': [{'perspective': 'back',
   'sizes': [{'size': 'xlarge',
     'url': 'https://www.kroger.com/product/images/xlarge/back/0001111079770'},
    {'size': 'large',
     'url': 'https://www.kroger.com/product/images/large/back/0001111079770'},
    {'size': 'medium',
     'url': 'https://www.kroger.com/product/images/medium/back/0001111079770'},
    {'size': 'small',
     'url': 'https://www.kroger.com/pr